In [1]:
import os, json, pathlib, re, ast
import openai
from tqdm import tqdm
import pandas as pd
from dotenv import load_dotenv
from typing import List, Dict, Optional
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.docstore.document import Document
from langchain_openai import OpenAIEmbeddings

In [2]:
# Load API
load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")
MODEL_NAME     = "gpt-4o-mini-2024-07-18" 
TEMPERATURE    = 1.0
print("API Key loaded:", openai.api_key is not None)

# Index file
INDEX_DIR = pathlib.Path("faiss_disaster_information_index")

# Load files
src_file  = "../data/disaster_description.csv"
out_file  = "extracted_rag.csv"
df = pd.read_csv(src_file)
# df = df.head(200)

API Key loaded: True


# FAISS Vectorstore

In [4]:
DISASTER_CLASSIFICATION = json.load(open("../data/classification_document.json"))

In [5]:
MAGNITUDE_INFORMATION = json.load(open("../data/magnitude_document.json"))

In [6]:
FIELD_SCHEMA = json.load(open("../data/schema_document.json"))

In [7]:
EXAMPLES = pathlib.Path("../data/Examples.txt").read_text(encoding="utf-8")

In [8]:
disaster_docs = []
groups = DISASTER_CLASSIFICATION["Disaster Group"]
for group_name, meta in groups.items():
        desc  = meta.get("Description", "")
        types = meta.get("Disaster Types", {})

        block = f"### Disaster Group: {group_name}\n{desc}\n\nDisaster Types:\n"
        for t_name, t_desc in types.items():
            block += f"- {t_name}: {t_desc}\n"

        doc = Document(
            page_content = block.strip(),
            metadata     = {"group": group_name, "level": "G1"}
        )
        disaster_docs.append(doc)

In [9]:
magnitude_docs = []
for dtype, info in MAGNITUDE_INFORMATION.items():
    block = (
        f"### Magnitude Guidance\n"
        f"Disaster Type: {dtype}\n"
        f"Property      : {info['property']}\n"
        f"Unit          : {info['unit']}"
    )

    doc = Document(
        page_content = block,
        metadata     = {"type" : dtype, "level": "M1" }
    )
    magnitude_docs.append(doc)

In [10]:
fields_docs = []
for section_name, fields in FIELD_SCHEMA.items():
    for fname, meta in fields.items():
        block = (
            f"### Field: {fname}\n"
            f"Section      : {section_name}\n"
            f"Type         : {meta['Type']}\n"
            f"Description  : {meta.get('Description', meta.get('Rescription',''))}"
        )

        doc = Document(
            page_content = block,
            metadata = {"section": section_name, "field"  : fname, "level"  : "F0"
                }
        )
        fields_docs.append(doc)

In [11]:
examples_docs = []
block = re.split(r"Example\s+\d+:\s*", EXAMPLES)[1:]

for idx, chunk in enumerate(block, start=1):
    doc = Document(
        page_content = chunk.strip(),
        metadata     = {"example_id": idx, "level": "E1"}
    )
    examples_docs.append(doc)

In [12]:
if INDEX_DIR.exists():
    vectorstore = FAISS.load_local(
        str(INDEX_DIR),
        OpenAIEmbeddings(model="text-embedding-3-small"),
        allow_dangerous_deserialization=True 
    )
else:
    print("Building FAISS index from scratch … ")
    
    embeddings  = OpenAIEmbeddings(model="text-embedding-3-small")
    vectorstore = FAISS.from_documents(disaster_docs, embeddings)
    vectorstore.add_documents(magnitude_docs)
    vectorstore.add_documents(fields_docs)
    vectorstore.add_documents(examples_docs)
    vectorstore.save_local(str(INDEX_DIR))

print(f"✅  Vectorstore ready, ntotal = {vectorstore.index.ntotal}")

✅  Vectorstore ready, ntotal = 41


# Prompt

In [14]:
OUTPUT_FIELDS = {
  "Event": "fields name",
  "Geographical Information": "fields name",
  "Effect": "fields name"
}

In [15]:
OUTPUT_VALUES = {
  "Event": {"fields": "values"},
  "Geographical Information": {"fields": "values"},
  "Effect": {"fields": "values"}
}

In [16]:
fields_extraction_prompt = f"""
        Please refer FIELD_SCHEMA first, then list every field that is
        mentioned (even once) or is marked as 'Required' in DISASTER_DESCRIPTION. 
        Output must contain field names only.
        Finally return a single JSON object that conforms precisely to OUTPUT_FIELDS.
        
        FIELD_SCHEMA:
        {json.dumps(FIELD_SCHEMA, ensure_ascii=False)}

        OUTPUT_FIELDS:
        {json.dumps(OUTPUT_FIELDS, ensure_ascii=False)}
    """

In [17]:
def values_extraction_prompt(
    fields_docs: List[Document], 
    categorical_values: Optional[Document]
) -> str:
    PROMPT = """
    Please extract all information regarding only the fields in EXTRACTED_FIELDS 
    from text DISASTER_DESCRIPTION (refering the information in FIELD_SCHEMA and using only 
    the categorical options in CATEGORICAL_VALUES). 
    Finally return a single JSON object that conforms precisely to OUTPUT_VALUES.
"""

    parts = ["FIELD_SCHEMA:"]
    parts += [d.page_content for d in fields_docs]

    parts += ["CATEGORICAL_VALUES:"]
    parts += [d.page_content for d in categorical_values] 

    parts += [
        "OUTPUT_VALUES:",
        json.dumps(OUTPUT_VALUES, ensure_ascii=False)
    ]

    return PROMPT + "\n\n" + "\n\n".join(parts)


# Implement

In [19]:
def call_llm(prompt_text: str, description_text: str, extracted_fields_text = None) -> dict:
    
    if extracted_fields_text is None:
        user_content = description_text or ""
    else:
        user_content = json.dumps({
            "DISASTER_DESCRIPTION": description_text,
            "EXTRACTED_FIELDS": extracted_fields_text
        }, ensure_ascii=False)
        
    try:
        resp = openai.chat.completions.create(
            model           = MODEL_NAME,
            messages        = [
                {"role": "system", "content": prompt_text},
                {"role": "user", "content": user_content}
            ],
            temperature     = TEMPERATURE,
            response_format = {"type": "json_object"}
        )
        return json.loads(resp.choices[0].message.content)
    except Exception as e:
        print("LLM Error:", e)
        return None

In [20]:
def process_record(description: str) -> dict:
    if not isinstance(description, str):
        return None 
    
    # Stage1: fields
    r1 = call_llm(fields_extraction_prompt, description)

    if not isinstance(r1, dict) or r1 is None:
        return None
    
    # Stage2: values
    # 2-1: fields_docs
    if not isinstance(r1, dict):
        print("The type of r1 is not dictionary:", r1)
        return None
    try:
        field_list = [fname for section in r1.values() for fname in section]
    except Exception as e:
        print("The values of r1 are not lists", r1)
        return None    
    
    f0_docs = []
    for fname in field_list:
        doc = vectorstore.similarity_search(
            description,
            k=1,
            filter={"level": "F0", "field": fname}
        )
        if doc:
            f0_docs.append(doc[0])
        else:
            # fallback
            field_doc_map = {d.metadata["field"]: d.page_content for d in fields_docs}
            f0_docs.append(
                Document(
                    page_content=field_doc_map.get(fname, ""),
                    metadata={"level": "F0", "field": fname}
                )
            )
    
    # 2-2: categorical values
    g1_docs = vectorstore.similarity_search(
        description,
        k=2,
        filter={"level": "G1"}
    )
    
    m1_doc = None
    if {"Magnitude", "Magnitude Scale"} & set(field_list):
        hits = vectorstore.similarity_search(
            description,
            k=1,
            filter={"level": "M1"}
        )
        m1_doc = hits[0] if hits else None

    categorical_values = g1_docs + ([m1_doc] if m1_doc else [])
    
    # 2-3: extraction
    p2 = values_extraction_prompt(
        fields_docs      = f0_docs,
        categorical_values = categorical_values
    )
    r2 = call_llm(p2, description, r1)
    
    if r2 is None:
        return None
    
    return {
        "RAG_fields": json.dumps(r1, ensure_ascii=False),
        "RAG_values": json.dumps(r2, ensure_ascii=False)
    }


In [21]:
results = []
for desc in tqdm(df["description"], desc="Processing"):
    out = process_record(desc)
    if out is None:
        results.append({"RAG_fields": None, "RAG_values": None})
    else:
        results.append(out)

df_out = pd.concat(
    [df.reset_index(drop=True), pd.DataFrame(results)],
    axis=1
)

df_out.to_csv(out_file, index=False)
print(f"Output saved to {out_file}")


Processing: 100%|█████████████████████████| 1588/1588 [3:10:13<00:00,  7.19s/it]

Output saved to extracted_rag.csv
